# Feature Engineering

Objective:
- Create advanced features for theft detection
- Generate time-series features
- Add transformer-level indicators
- Improve model performance


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)


In [2]:
df = pd.read_csv("../data/processed/feature_engineered.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

df.head()


,consumer_id,timestamp,kwh,voltage,current,power_factor,theft_label,transformer_id,total_kwh,temperature,humidity,hour,day,month,day_of_week,load_ratio,rolling_mean_24h,rolling_std_24h
0,1,2024-01-01 00:00:00,1.408501,231.134680,6.600922,0.970170,0,T7,46.629769,25.086067,55.189874,0,1,1,0,0.030206,1.408501,0.000000
1,1,2024-01-01 01:00:00,1.590901,226.709743,6.022453,0.949561,0,T7,49.026706,25.086067,79.649722,1,1,1,0,0.032450,1.499701,0.128977
2,1,2024-01-01 02:00:00,1.539829,233.289587,6.373323,0.967279,0,T7,50.220320,25.086067,63.687502,2,1,1,0,0.030661,1.513077,0.094097
3,1,2024-01-01 03:00:00,1.215253,233.507623,5.297672,0.879270,0,T7,53.799312,25.086067,76.549985,3,1,1,0,0.022589,1.438621,0.167564
4,1,2024-01-01 04:00:00,2.131333,231.189122,8.212522,0.987504,0,T7,54.309090,25.086067,55.590191,4,1,1,0,0.039244,1.577163,0.342094


In [3]:
df["is_weekend"] = df["day_of_week"].apply(lambda x: 1 if x >= 5 else 0)
df["is_night"] = df["hour"].apply(lambda x: 1 if x <= 5 else 0)

df.head()


,consumer_id,timestamp,kwh,voltage,current,power_factor,theft_label,transformer_id,total_kwh,temperature,humidity,hour,day,month,day_of_week,load_ratio,rolling_mean_24h,rolling_std_24h,is_weekend,is_night
0,1,2024-01-01 00:00:00,1.408501,231.134680,6.600922,0.970170,0,T7,46.629769,25.086067,55.189874,0,1,1,0,0.030206,1.408501,0.000000,0,1
1,1,2024-01-01 01:00:00,1.590901,226.709743,6.022453,0.949561,0,T7,49.026706,25.086067,79.649722,1,1,1,0,0.032450,1.499701,0.128977,0,1
2,1,2024-01-01 02:00:00,1.539829,233.289587,6.373323,0.967279,0,T7,50.220320,25.086067,63.687502,2,1,1,0,0.030661,1.513077,0.094097,0,1
3,1,2024-01-01 03:00:00,1.215253,233.507623,5.297672,0.879270,0,T7,53.799312,25.086067,76.549985,3,1,1,0,0.022589,1.438621,0.167564,0,1
4,1,2024-01-01 04:00:00,2.131333,231.189122,8.212522,0.987504,0,T7,54.309090,25.086067,55.590191,4,1,1,0,0.039244,1.577163,0.342094,0,1


In [4]:
df = df.sort_values(["consumer_id", "timestamp"])

df["lag_1"] = df.groupby("consumer_id")["kwh"].shift(1)
df["lag_24"] = df.groupby("consumer_id")["kwh"].shift(24)

df[["consumer_id","timestamp","kwh","lag_1","lag_24"]].head(30)


,consumer_id,timestamp,kwh,lag_1,lag_24
0,1,2024-01-01 00:00:00,1.408501,NaN,NaN
1,1,2024-01-01 01:00:00,1.590901,1.408501,NaN
2,1,2024-01-01 02:00:00,1.539829,1.590901,NaN
3,1,2024-01-01 03:00:00,1.215253,1.539829,NaN
4,1,2024-01-01 04:00:00,2.131333,1.215253,NaN
5,1,2024-01-01 05:00:00,1.682347,2.131333,NaN
6,1,2024-01-01 06:00:00,1.809626,1.682347,NaN
7,1,2024-01-01 07:00:00,1.822963,1.809626,NaN
8,1,2024-01-01 08:00:00,1.941002,1.822963,NaN
9,1,2024-01-01 09:00:00,1.929731,1.941002,NaN


In [5]:
df["diff_1"] = df["kwh"] - df["lag_1"]
df["diff_24"] = df["kwh"] - df["lag_24"]

df.head()


,consumer_id,timestamp,kwh,voltage,current,power_factor,theft_label,transformer_id,total_kwh,temperature,humidity,hour,day,month,day_of_week,load_ratio,rolling_mean_24h,rolling_std_24h,is_weekend,is_night,lag_1,lag_24,diff_1,diff_24
0,1,2024-01-01 00:00:00,1.408501,231.134680,6.600922,0.970170,0,T7,46.629769,25.086067,55.189874,0,1,1,0,0.030206,1.408501,0.000000,0,1,NaN,NaN,NaN,NaN
1,1,2024-01-01 01:00:00,1.590901,226.709743,6.022453,0.949561,0,T7,49.026706,25.086067,79.649722,1,1,1,0,0.032450,1.499701,0.128977,0,1,1.408501,NaN,0.182400,NaN
2,1,2024-01-01 02:00:00,1.539829,233.289587,6.373323,0.967279,0,T7,50.220320,25.086067,63.687502,2,1,1,0,0.030661,1.513077,0.094097,0,1,1.590901,NaN,-0.051072,NaN
3,1,2024-01-01 03:00:00,1.215253,233.507623,5.297672,0.879270,0,T7,53.799312,25.086067,76.549985,3,1,1,0,0.022589,1.438621,0.167564,0,1,1.539829,NaN,-0.324576,NaN
4,1,2024-01-01 04:00:00,2.131333,231.189122,8.212522,0.987504,0,T7,54.309090,25.086067,55.590191,4,1,1,0,0.039244,1.577163,0.342094,0,1,1.215253,NaN,0.916079,NaN


In [6]:
df["z_score"] = (
    (df["kwh"] - df["rolling_mean_24h"]) /
    (df["rolling_std_24h"] + 1e-6)
)

df.head()


,consumer_id,timestamp,kwh,voltage,current,power_factor,theft_label,transformer_id,total_kwh,temperature,humidity,hour,day,month,day_of_week,load_ratio,rolling_mean_24h,rolling_std_24h,is_weekend,is_night,lag_1,lag_24,diff_1,diff_24,z_score
0,1,2024-01-01 00:00:00,1.408501,231.134680,6.600922,0.970170,0,T7,46.629769,25.086067,55.189874,0,1,1,0,0.030206,1.408501,0.000000,0,1,NaN,NaN,NaN,NaN,0.000000
1,1,2024-01-01 01:00:00,1.590901,226.709743,6.022453,0.949561,0,T7,49.026706,25.086067,79.649722,1,1,1,0,0.032450,1.499701,0.128977,0,1,1.408501,NaN,0.182400,NaN,0.707101
2,1,2024-01-01 02:00:00,1.539829,233.289587,6.373323,0.967279,0,T7,50.220320,25.086067,63.687502,2,1,1,0,0.030661,1.513077,0.094097,0,1,1.590901,NaN,-0.051072,NaN,0.284302
3,1,2024-01-01 03:00:00,1.215253,233.507623,5.297672,0.879270,0,T7,53.799312,25.086067,76.549985,3,1,1,0,0.022589,1.438621,0.167564,0,1,1.539829,NaN,-0.324576,NaN,-1.333025
4,1,2024-01-01 04:00:00,2.131333,231.189122,8.212522,0.987504,0,T7,54.309090,25.086067,55.590191,4,1,1,0,0.039244,1.577163,0.342094,0,1,1.215253,NaN,0.916079,NaN,1.619930


In [7]:
# Consumer share compared to average share
transformer_avg_share = df.groupby("transformer_id")["load_ratio"].transform("mean")

df["transformer_deviation"] = df["load_ratio"] - transformer_avg_share

df.head()


,consumer_id,timestamp,kwh,voltage,current,power_factor,theft_label,transformer_id,total_kwh,temperature,humidity,hour,day,month,day_of_week,load_ratio,rolling_mean_24h,rolling_std_24h,is_weekend,is_night,lag_1,lag_24,diff_1,diff_24,z_score,transformer_deviation
0,1,2024-01-01 00:00:00,1.408501,231.134680,6.600922,0.970170,0,T7,46.629769,25.086067,55.189874,0,1,1,0,0.030206,1.408501,0.000000,0,1,NaN,NaN,NaN,NaN,0.000000,-0.008255
1,1,2024-01-01 01:00:00,1.590901,226.709743,6.022453,0.949561,0,T7,49.026706,25.086067,79.649722,1,1,1,0,0.032450,1.499701,0.128977,0,1,1.408501,NaN,0.182400,NaN,0.707101,-0.006012
2,1,2024-01-01 02:00:00,1.539829,233.289587,6.373323,0.967279,0,T7,50.220320,25.086067,63.687502,2,1,1,0,0.030661,1.513077,0.094097,0,1,1.590901,NaN,-0.051072,NaN,0.284302,-0.007800
3,1,2024-01-01 03:00:00,1.215253,233.507623,5.297672,0.879270,0,T7,53.799312,25.086067,76.549985,3,1,1,0,0.022589,1.438621,0.167564,0,1,1.539829,NaN,-0.324576,NaN,-1.333025,-0.015873
4,1,2024-01-01 04:00:00,2.131333,231.189122,8.212522,0.987504,0,T7,54.309090,25.086067,55.590191,4,1,1,0,0.039244,1.577163,0.342094,0,1,1.215253,NaN,0.916079,NaN,1.619930,0.000783


In [8]:
df["is_peak_hour"] = df["hour"].apply(lambda x: 1 if 18 <= x <= 22 else 0)

df.head()


,consumer_id,timestamp,kwh,voltage,current,power_factor,theft_label,transformer_id,total_kwh,temperature,humidity,hour,day,month,day_of_week,load_ratio,rolling_mean_24h,rolling_std_24h,is_weekend,is_night,lag_1,lag_24,diff_1,diff_24,z_score,transformer_deviation,is_peak_hour
0,1,2024-01-01 00:00:00,1.408501,231.134680,6.600922,0.970170,0,T7,46.629769,25.086067,55.189874,0,1,1,0,0.030206,1.408501,0.000000,0,1,NaN,NaN,NaN,NaN,0.000000,-0.008255,0
1,1,2024-01-01 01:00:00,1.590901,226.709743,6.022453,0.949561,0,T7,49.026706,25.086067,79.649722,1,1,1,0,0.032450,1.499701,0.128977,0,1,1.408501,NaN,0.182400,NaN,0.707101,-0.006012,0
2,1,2024-01-01 02:00:00,1.539829,233.289587,6.373323,0.967279,0,T7,50.220320,25.086067,63.687502,2,1,1,0,0.030661,1.513077,0.094097,0,1,1.590901,NaN,-0.051072,NaN,0.284302,-0.007800,0
3,1,2024-01-01 03:00:00,1.215253,233.507623,5.297672,0.879270,0,T7,53.799312,25.086067,76.549985,3,1,1,0,0.022589,1.438621,0.167564,0,1,1.539829,NaN,-0.324576,NaN,-1.333025,-0.015873,0
4,1,2024-01-01 04:00:00,2.131333,231.189122,8.212522,0.987504,0,T7,54.309090,25.086067,55.590191,4,1,1,0,0.039244,1.577163,0.342094,0,1,1.215253,NaN,0.916079,NaN,1.619930,0.000783,0


In [9]:
df["rolling_max_24h"] = (
    df.groupby("consumer_id")["kwh"]
    .transform(lambda x: x.rolling(24, min_periods=1).max())
)

df["rolling_min_24h"] = (
    df.groupby("consumer_id")["kwh"]
    .transform(lambda x: x.rolling(24, min_periods=1).min())
)

df.head()


,consumer_id,timestamp,kwh,voltage,current,power_factor,theft_label,transformer_id,total_kwh,temperature,humidity,hour,day,month,day_of_week,load_ratio,rolling_mean_24h,rolling_std_24h,is_weekend,is_night,lag_1,lag_24,diff_1,diff_24,z_score,transformer_deviation,is_peak_hour,rolling_max_24h,rolling_min_24h
0,1,2024-01-01 00:00:00,1.408501,231.134680,6.600922,0.970170,0,T7,46.629769,25.086067,55.189874,0,1,1,0,0.030206,1.408501,0.000000,0,1,NaN,NaN,NaN,NaN,0.000000,-0.008255,0,1.408501,1.408501
1,1,2024-01-01 01:00:00,1.590901,226.709743,6.022453,0.949561,0,T7,49.026706,25.086067,79.649722,1,1,1,0,0.032450,1.499701,0.128977,0,1,1.408501,NaN,0.182400,NaN,0.707101,-0.006012,0,1.590901,1.408501
2,1,2024-01-01 02:00:00,1.539829,233.289587,6.373323,0.967279,0,T7,50.220320,25.086067,63.687502,2,1,1,0,0.030661,1.513077,0.094097,0,1,1.590901,NaN,-0.051072,NaN,0.284302,-0.007800,0,1.590901,1.408501
3,1,2024-01-01 03:00:00,1.215253,233.507623,5.297672,0.879270,0,T7,53.799312,25.086067,76.549985,3,1,1,0,0.022589,1.438621,0.167564,0,1,1.539829,NaN,-0.324576,NaN,-1.333025,-0.015873,0,1.590901,1.215253
4,1,2024-01-01 04:00:00,2.131333,231.189122,8.212522,0.987504,0,T7,54.309090,25.086067,55.590191,4,1,1,0,0.039244,1.577163,0.342094,0,1,1.215253,NaN,0.916079,NaN,1.619930,0.000783,0,2.131333,1.215253


In [10]:
df["sudden_drop_flag"] = df["diff_1"].apply(lambda x: 1 if x < -1 else 0)

df.head()


,consumer_id,timestamp,kwh,voltage,current,power_factor,theft_label,transformer_id,total_kwh,temperature,humidity,hour,day,month,day_of_week,load_ratio,rolling_mean_24h,rolling_std_24h,is_weekend,is_night,lag_1,lag_24,diff_1,diff_24,z_score,transformer_deviation,is_peak_hour,rolling_max_24h,rolling_min_24h,sudden_drop_flag
0,1,2024-01-01 00:00:00,1.408501,231.134680,6.600922,0.970170,0,T7,46.629769,25.086067,55.189874,0,1,1,0,0.030206,1.408501,0.000000,0,1,NaN,NaN,NaN,NaN,0.000000,-0.008255,0,1.408501,1.408501,0
1,1,2024-01-01 01:00:00,1.590901,226.709743,6.022453,0.949561,0,T7,49.026706,25.086067,79.649722,1,1,1,0,0.032450,1.499701,0.128977,0,1,1.408501,NaN,0.182400,NaN,0.707101,-0.006012,0,1.590901,1.408501,0
2,1,2024-01-01 02:00:00,1.539829,233.289587,6.373323,0.967279,0,T7,50.220320,25.086067,63.687502,2,1,1,0,0.030661,1.513077,0.094097,0,1,1.590901,NaN,-0.051072,NaN,0.284302,-0.007800,0,1.590901,1.408501,0
3,1,2024-01-01 03:00:00,1.215253,233.507623,5.297672,0.879270,0,T7,53.799312,25.086067,76.549985,3,1,1,0,0.022589,1.438621,0.167564,0,1,1.539829,NaN,-0.324576,NaN,-1.333025,-0.015873,0,1.590901,1.215253,0
4,1,2024-01-01 04:00:00,2.131333,231.189122,8.212522,0.987504,0,T7,54.309090,25.086067,55.590191,4,1,1,0,0.039244,1.577163,0.342094,0,1,1.215253,NaN,0.916079,NaN,1.619930,0.000783,0,2.131333,1.215253,0


In [11]:
df = df.fillna(0)

df.isnull().sum().sum()


np.int64(0)

In [12]:
feature_columns = [
    "kwh",
    "voltage",
    "current",
    "power_factor",
    "temperature",
    "humidity",
    "hour",
    "day",
    "month",
    "day_of_week",
    "is_weekend",
    "is_night",
    "is_peak_hour",
    "load_ratio",
    "rolling_mean_24h",
    "rolling_std_24h",
    "lag_1",
    "lag_24",
    "diff_1",
    "diff_24",
    "z_score",
    "transformer_deviation",
    "rolling_max_24h",
    "rolling_min_24h",
    "sudden_drop_flag"
]

print("Total Features:", len(feature_columns))


Total Features: 25


In [13]:
df.to_csv("../data/processed/feature_engineered_v2.csv", index=False)

print("Enhanced dataset saved successfully.")


Enhanced dataset saved successfully.


## Feature Engineering Summary

We created:

1. Lag features (1 hour, 24 hours)
2. Rolling statistics
3. Behavioral z-score
4. Transformer deviation
5. Sudden drop indicators
6. Peak hour & weekend flags

These features capture:
- Temporal patterns
- Behavioral deviation
- Grid-level anomalies
- Suspicious consumption shifts

Next Step:
- Train Anomaly Detection Model
- Train Supervised Classification Model
